# A2C

In [1]:
from mountaincar_utils import test_car, env_mountaincar, display_frames_as_gif, ReplayMemory
from IPython.display import HTML

In [2]:
# networks
import torch.nn as nn
import torch

class ValueNet(nn.Module):
    def __init__(self, num_states=4):
        super(ValueNet, self).__init__()
        self.fc1 = nn.Linear(num_states, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)
    
class ActorNet(nn.Module):
    def __init__(self, num_states=2):
        super(ActorNet, self).__init__()
        self.num_states = num_states
        self.shared = nn.Sequential(nn.Linear(self.num_states, 64), nn.ReLU())
        self.mean_layer = nn.Linear(64, 1)
        self.log_std_layer = nn.Linear(64, 1)

    def forward(self, state):
        x = self.shared(state)
        # Binds mean action to [-1.0, 1.0]
        mean = torch.tanh(self.mean_layer(x)) 
        # Keeps standard deviation positive
        std = torch.exp(torch.clamp(self.log_std_layer(x), -20.0, 2.0)) 
        return mean, std

In [6]:
import torch.nn.functional as F
import numpy as np
from torch.distributions import Normal

class A2CAgent:
    def __init__(self):
        self.value_net = ValueNet(num_states=2)
        self.actor_net = ActorNet(num_states=2)
        self.opt_value = torch.optim.AdamW(self.value_net.parameters(), lr=0.001)
        self.opt_actor= torch.optim.AdamW(self.actor_net.parameters(), lr=0.001)
        self.action_value = 0.0
        self.log_prob = -1.0
        self.action_delta = 0.0

    def reset(self):
        self.action_value = 0.0
        self.log_prob = -1.0
        self.action_delta = 0.0

    def act(self, state, train=True):
        if isinstance(state, np.ndarray):
            if len(state.shape) == 1:
                state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            else:
                state = torch.tensor(state, dtype=torch.float32)
        with torch.no_grad():
            state = state * torch.tensor([1.0, 2.0], dtype=torch.float32)
            mean, std = self.actor_net(state)
            dist = Normal(mean.detach(), std.detach())
            self.action_delta = dist.sample().detach().clone().item()
            # self.log_prob = dist.log_prob(torch.tensor(self.action, dtype=torch.float32))

        # convert the scalar action to the shape expected by MountainCarContinuous
        # env_input = (self.action * 2.0) - 1.0

        self.action_value = np.clip(self.action_value + self.action_delta, -1.0, 1.0)
        env_input = np.array([self.action_value], dtype=np.float32)

        return env_input
    
    def learn(self, plays):
        def cum_reward(rewards, gamma=0.99):
            cum_rewards = torch.zeros_like(rewards).to(torch.float32)
            for j in range(len(rewards))[::-1]:
                cum_rewards[j] = rewards[j] + gamma * (cum_rewards[j+1] if j+1<len(rewards) else 0)
            eps = np.finfo(np.float32).eps.item()
            cum_rewards = (cum_rewards - cum_rewards.mean()) / (cum_rewards.std() + eps)
            return cum_rewards

        # get data
        samples = plays.sample()
        
        # get samples batch
        states, rewards, actions, _ = samples

        # compute cumulated reward
        rewards_cum = cum_reward(rewards)

        # critic
        values = self.value_net(states)
        vloss = F.mse_loss(values, rewards_cum, reduction='sum')
        self.opt_value.zero_grad()
        vloss.backward()
        self.opt_value.step()

        # actor
        with torch.no_grad():
            values = self.value_net(states).detach()
        advantages = rewards_cum - values

        outputs = self.actor_net(states)
        dists = Normal(outputs[0], outputs[1])
        log_probs = dists.log_prob(actions)
        
        aloss = (-log_probs * advantages).sum()
        self.opt_actor.zero_grad()
        aloss.backward()
        self.opt_actor.step()

        return vloss.detach().item(), aloss.detach().item()

In [7]:
# train loop
from collections import deque

epochs = 1000
state_num = 4 # 
action_num = 2
memory = ReplayMemory()
agent = A2CAgent()

scores = []
vlosses, alosses = [], []
recent_scores = deque(maxlen=100)

for e in range(epochs):
    # reset environment
    state, _ = env_mountaincar.reset()
    agent.reset() 

    currState = state
    done = False

    score = 0
    tot_loss = 0
    count = 0
    tot_reward = 0

    # run an episode
    while not done :
        
        # choose action
        action = agent.act(state)

        # take action on env
        state, reward, terminated, truncated, info = env_mountaincar.step(action)
        done = terminated or truncated   
        
        # add to replay memory
        memory.add([currState, reward, agent.action_delta, done])

        currState = state.copy()

        # update score
        score += 1
        tot_reward += reward

        if score > 800:
            reward = -100.0  # Give a large positive reward for reaching the goal
            break
    
    # train
    vloss, aloss = agent.learn(memory)

    # clear memory
    memory.clear()
    
    scores = np.append(scores, score)
    vlosses = np.append(vlosses, vloss)
    alosses = np.append(alosses, aloss)
    recent_scores.append(tot_reward)

    if (e+1)%100 == 0:
        print(f"epoch: {e+1}, reward: {tot_reward}, score: {score}, value loss: {vloss:.6f}, actor loss: {aloss:.6f}")
    # early stopping if the goal is reached
    if len(recent_scores) >= 100:
        average_reward = sum(recent_scores) / 100
        if average_reward > 80.:
            print(f"Early stopping at episode {e+1} with average reward: {average_reward:.2f}")
            break

epoch: 100, reward: -54.612111876782286, score: 801, value loss: 770.266907, actor loss: 67.562881
epoch: 200, reward: -52.77615284092762, score: 801, value loss: 617.435181, actor loss: 39.238995
epoch: 300, reward: 65.17924761546475, score: 536, value loss: 346.854492, actor loss: -59.524643
epoch: 400, reward: 53.74133661000731, score: 615, value loss: 371.349457, actor loss: -18.062675
epoch: 500, reward: 52.70424377052866, score: 675, value loss: 505.466156, actor loss: -107.692596
epoch: 600, reward: 57.53004475434922, score: 572, value loss: 394.531342, actor loss: -61.623386
epoch: 700, reward: 79.07263667153313, score: 305, value loss: 193.126556, actor loss: -34.525795
epoch: 800, reward: 81.75511406315358, score: 284, value loss: 90.080185, actor loss: 6.950966
epoch: 900, reward: 78.99499689975416, score: 353, value loss: 215.571686, actor loss: 13.098902
epoch: 1000, reward: 84.74121452621611, score: 220, value loss: 134.667740, actor loss: -36.393150


In [9]:
frames = test_car(env_mountaincar, 900, agent=agent)
anim = display_frames_as_gif(frames)
HTML(anim.to_jshtml())

494
